# บทที่ 18: การรักษาความปลอดภัยเอเจนต์ AI ด้วยใบเสร็จเชิงเข้ารหัส

## สมุดบันทึกแบบปฏิบัติ

สมุดบันทึกนี้จะพาคุณทำผ่านสี่แบบฝึกหัด:

1. **เซ็นใบเสร็จของคุณครั้งแรก** สำหรับการเรียกใช้เครื่องมือเอเจนต์และตรวจสอบความถูกต้อง
2. **ดัดแปลงใบเสร็จ** และสังเกตการตรวจสอบล้มเหลว
3. **สร้างโซ่ใบเสร็จสามใบ** และยืนยันความสมบูรณ์ของโซ่
4. **ห่อหุ้มการเรียกใช้เครื่องมือ Microsoft Agent Framework** เพื่อให้ทุกการกระทำสร้างใบเสร็จ

ทุกพื้นฐานเชิงเข้ารหัสถูกนำเข้าจากไลบรารีที่ดูแลอย่างดี (`pynacl` สำหรับ Ed25519, `jcs` สำหรับ RFC 8785 canonical JSON, `hashlib` จากไลบรารีมาตรฐานของ Python สำหรับ SHA-256) ตรรกะของใบเสร็จเองเป็น Python ธรรมดาที่คุณสามารถอ่านและแก้ไขได้

รันโค้ดในแต่ละช่องตามลำดับ ทุกส่วนสั้นและแยกตัว


## การติดตั้ง

ติดตั้ง dependencies สองตัว ทั้งสองตัวมีใบอนุญาตแบบเสรี (Apache-2.0 / MIT)


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## เครื่องมือช่วยเหลือ

เครื่องมือช่วยเหลือสองตัวนี้จัดการกับการเข้ารหัส base64url (โดยไม่ใส่ padding) และการแฮช SHA-256 ของวัตถุใดๆ ก็ตาม เพื่อให้ส่วนที่เหลือของสมุดบันทึกมุ่งเน้นไปที่ตรรกะของใบเสร็จโดยตรง


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: เซ็นใบเสร็จครั้งแรกของคุณ

ลองจินตนาการว่าตัวแทนของเราใน **Contoso Travel** เพิ่งค้นหาเที่ยวบินจากซิดนีย์ไปลอสแองเจลิสให้กับลูกค้า เราต้องการบันทึกการเรียกใช้เครื่องมือนี้เป็นใบเสร็จที่เซ็นแล้ว เพื่อให้นักตรวจสอบในอนาคตสามารถตรวจสอบได้โดยไม่ต้องไว้ใจเรา

### ขั้นตอนที่ 1.1: สร้างกุญแจเซ็นชื่อ

ในการใช้งานจริง กุญแจเซ็นของตัวแทนจะถูกเก็บไว้ในฮาร์ดแวร์เซฟตี้โมดูล (HSM), Azure Key Vault หรือที่เก็บข้อมูลที่ปลอดภัยแบบเดียวกัน สำหรับบทเรียนนี้เราจะสร้างกุญแจใหม่ในหน่วยความจำ


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### ขั้นตอนที่ 1.2: สร้าง payload ของใบเสร็จ

Payload ประกอบด้วยทุกสิ่งที่เราต้องการให้ใบเสร็จรับรอง: ใครเป็นผู้กระทำ, เครื่องมืออะไร, พร้อมด้วยอาร์กิวเมนต์อะไร, ผลลัพธ์ที่กลับมา, ภายใต้นโยบายอะไร, และเมื่อใด เราจะทำการแฮชอาร์กิวเมนต์และผลลัพธ์แทนที่จะรวมไว้ในข้อความโดยตรงเพื่อที่ใบเสร็จจะได้ไม่รั่วไหลเนื้อหาที่ละเอียดอ่อน


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### ขั้นตอนที่ 1.3: ลงนามและประกอบใบเสร็จ

สามขั้นตอน:

1. ทำให้ข้อมูลเพย์โหลดเป็นมาตรฐานโดยใช้ JCS เพื่อให้สองการใช้งานที่สร้างใบเสร็จที่มีความหมายเหมือนกันสร้างไบต์ที่เหมือนกันแบบไบต์ต่อไบต์
2. แฮชไบต์มาตรฐานด้วย SHA-256
3. ลงนามแฮชด้วยคีย์ส่วนตัว Ed25519

จากนั้นลายเซ็นจะถูกแนบไปกับเพย์โหลดต้นฉบับเพื่อสร้างใบเสร็จสุดท้าย


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### ขั้นตอนที่ 1.4: ตรวจสอบใบเสร็จรับเงิน

การตรวจสอบคือการย้อนกระบวนการ เราจะลบลายเซ็น คำนวณแฮชแคนอนิคอลใหม่ และตรวจสอบลายเซ็นกับกุญแจสาธารณะในใบเสร็จรับเงิน

ผู้ตรวจสอบที่ทำการตรวจสอบนี้ไม่ต้องการสิ่งใดจากเรานอกจากใบเสร็จรับเงินเท่านั้น ไม่มีบริการให้เรียก ไม่มีไดเรกทอรีคีย์ให้สอบถาม และไม่ต้องมีความน่าเชื่อถือ


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

คุณควรเห็นข้อความ `Receipt is valid: True` ตัวแทนได้สร้างบันทึกตรวจสอบที่ลงนามทางคริปโตกราฟีเป็นครั้งแรก


## ส่วนที่ 2: ปลอมแปลงใบเสร็จ

จุดประสงค์ของใบเสร็จทั้งหมดคือเพื่อให้สามารถตรวจจับการปลอมแปลงได้ มาแสดงให้เห็น

เราจะเปลี่ยนแปลงตัวอักษรเพียงตัวเดียวในใบเสร็จและดูการตรวจสอบล้มเหลว


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### เกิดอะไรขึ้นเมื่อกี้นี้?

เมื่อเราทำการเปลี่ยนแปลง `policy_id` ไบต์มาตรฐานจึงเปลี่ยนไป ค่าแฮช SHA-256 ของไบต์เหล่านั้นก็เปลี่ยนแปลงไป ลายเซ็น (ซึ่งอยู่บนแฮชต้นฉบับ) จึงไม่ตรงกับแฮชใหม่ การตรวจสอบจึงคืนค่า `False` อย่างถูกต้อง

ไม่มีวิธีใดที่จะปรับแต่งฟิลด์ใด ๆ ของใบเสร็จแล้วยังคงตรวจสอบได้ นอกจากผู้โจมตีจะมีคีย์ส่วนตัว ในขณะที่คีย์ส่วนตัวเก็บอยู่ในคลังคีย์และคีย์สาธารณะเผยแพร่ การปลอมแปลงจึงเป็นไปไม่ได้ที่จะซ่อน

ลองทำด้วยตัวเอง: ปรับเปลี่ยน `tool_name` หรือ `agent_id` หรือ `timestamp` ในเซลล์ด้านบนแล้วรันใหม่ ทุกการเปลี่ยนแปลงจะทำให้ใบเสร็จเป็นโมฆะ


## ส่วนที่ 3: เชื่อมใบเสร็จเข้าด้วยกันเป็นสายโซ่

ใบเสร็จหนึ่งใบคุ้มครองการกระทำหนึ่งอย่าง ตัวแทนส่วนใหญ่ทำการกระทำหลายอย่าง เพื่อทำให้ลำดับทั้งหมดตรวจจับการปลอมแปลงได้ เราจึงเชื่อมใบเสร็จแต่ละใบเข้ากับใบเสร็จใบก่อนหน้าด้วยการรวมค่าแฮชของใบเสร็จใบก่อนหน้าไว้ใน payload ของใบเสร็จใหม่

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

ถ้าใครลบหรือเรียงใบเสร็จใหม่ โซ่จะขาดที่จุดนั้นพอดี การตรวจสอบใบเสร็จใบใดใบหนึ่งที่ตามมาจะล้มเหลวเพราะ `previous_receipt_hash` ไม่ตรงกับค่าแฮชจริงของใบเสร็จที่มาก่อนหน้า


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

ตอนนี้ทำลายโซ่โดยแก้ไขใบเสร็จตรงกลางแล้วตรวจสอบใหม่ ใบเสร็จที่ถูกแก้ไขไม่ผ่านการตรวจสอบลายเซ็น และใบเสร็จถัดไปไม่ผ่านการตรวจสอบการเชื่อมโยงโซ่ (เพราะ `previous_receipt_hash` ของมันไม่ตรงกับค่าแฮชของใบเสร็จตรงกลางที่ถูกแก้ไขแล้ว)


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

ใบเสร็จ 0 ยังคงตรวจสอบได้ (ไม่ถูกแก้ไขและไม่มีใบเสร็จก่อนหน้าที่ต้องพึ่งพา) ใบเสร็จ 1 ล้มเหลวในการตรวจสอบลายเซ็นเนื่องจากเราเปลี่ยน `tool_args_hash` ใบเสร็จ 2 ล้มเหลวในการตรวจสอบลูกโซ่เพราะ `previous_receipt_hash` ของมันถูกคำนวณกับใบเสร็จ 1 ที่เดิม (ซึ่งตอนนี้ถูกแก้ไขแล้ว)

แม้ว่าผู้โจมตีจะลงลายเซ็นใบเสร็จ 1 ที่ถูกแก้ไขใหม่ (ซึ่งพวกเขาทำไม่ได้โดยไม่มีคีย์ส่วนตัว) ความไม่สอดคล้องของลูกโซ่ในใบเสร็จ 2 จะยังเปิดเผยการปลอมแปลง เพื่อซ่อนการเปลี่ยนแปลงนั้น ผู้โจมตีต้องลงลายเซ็นในทุกใบเสร็จตั้งแต่จุดที่แก้ไขเป็นต้นไป ซึ่งต้องการการครอบครองคีย์ส่วนตัว


## ส่วนที่ 4: ห่อการเรียกใช้เครื่องมือของเอเย่นต์ด้วยการลงนามใบเสร็จ

ในการใช้งานจริง คุณไม่ต้องการให้ผู้เขียนเอเย่นต์ทุกคนจดจำการเรียกใช้ `make_receipt` คุณต้องการให้การลงนามใบเสร็จเป็นไปโดยอัตโนมัติสำหรับการเรียกใช้เครื่องมือทุกครั้ง

นี่คือลักษณะรูปแบบที่ง่ายที่สุด: คลาสตัวห่อที่รับฟังก์ชันเครื่องมือที่เรียกได้ใด ๆ และส่งกลับเวอร์ชันที่สร้างใบเสร็จ นี่สามารถปรับใช้กับเฟรมเวิร์กเอเย่นต์ใด ๆ รวมถึง Microsoft Agent Framework (`agent_framework.foundry`)

หากคุณยังไม่ได้ตั้งค่าโปรเจกต์ Microsoft Foundry โมคในเครื่องด้านล่างก็ยังแสดงรูปแบบนี้ได้


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### การรวมเข้ากับ Microsoft Agent Framework

ตัว wrapper `ReceiptedTool` ข้างต้นไม่ขึ้นกับ framework ใดๆ ในการใช้มันภายในตัว agent ที่สร้างด้วย Microsoft Agent Framework ให้ลงทะเบียนฟังก์ชันที่ถูก wrapper เป็นเครื่องมือ หน้าตาคร่าว ๆ (คุณจะต้องเปลี่ยน mock เป็นการลงทะเบียนเครื่องมือ Microsoft Foundry จริง):

```python
# โค้ดจำลองแสดงรูปร่างของการผสานรวม
# นำเข้า os
# จาก agent_framework.foundry นำเข้า FoundryChatClient
# จาก azure.identity นำเข้า AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="คุณเป็นตัวแทนของ Contoso Travel ..."
#     tools=[receipted_lookup],   # เครื่องมือที่ถูกห่อหุ้ม ไม่ใช่ฟังก์ชันดิบ
# )
# response = agent.run("ค้นหาเที่ยวบินจากซิดนีย์ไปยังลอสแองเจลิสในเดือนมิถุนายน")
#
# # หลังการรัน ทุกการเรียกใช้เครื่องมือที่เอเจนต์ทำจะมีใบเสร็จลงนาม:
# audit_chain = receipted_lookup.receipts
```

framework ของ agent ไม่จำเป็นต้องรู้จักอะไรเกี่ยวกับใบเสร็จ การเซ็นชื่อใบเสร็จจะถูกรวมอยู่รอบ ๆ เครื่องมือ ไม่ได้ฝังเข้าไปใน framework นี่คือวิธีที่คุณเพิ่มที่มาที่ไปให้กับโค้ด agent ที่มีอยู่โดยไม่ต้องเขียนโค้ด agent ใหม่


## สรุปและความท้าทายเสริม

คุณได้:

- สร้างคู่กุญแจ Ed25519
- สร้างและลงนามใบเสร็จสำหรับการเรียกใช้เครื่องมือของเอเจนต์
- ตรวจสอบใบเสร็จแบบออฟไลน์โดยใช้เพียงกุญแจสาธารณะ
- ดัดแปลงใบเสร็จและสังเกตว่าการตรวจสอบล้มเหลว
- สร้างลำดับใบเสร็จที่เชื่อมด้วยแฮชสามใบเสร็จ
- ดัดแปลงตรงกลางของสายโซ่และสังเกตว่าทั้งลายเซ็นและการเชื่อมโยงในสายโซ่ล้มเหลว
- ห่อหุ้มฟังก์ชันเครื่องมือด้วยการเซ็นใบเสร็จโดยอัตโนมัติ

**ความท้าทายเสริม.** ขยายสคีม่าใบเสร็จด้วยฟิลด์ `request_id` (UUID สำหรับการติดตามแบบกระจาย) อัปเดต `make_receipt` เพื่อรวมฟิลด์นี้ และยืนยันว่าใบเสร็จยังคงตรวจสอบได้ครบถ้วน จากนั้นแก้ไขฟิลด์หลังการเซ็นและยืนยันว่าการตรวจสอบล้มเหลว ซึ่งจะบังคับให้คุณเข้าใจว่าแต่ละไบต์ของการเข้ารหัส canonical มีส่วนต่อการลงลายเซ็นอย่างไร

**ขอบเขตสำคัญ.** ใบเสร็จพิสูจน์สามสิ่งและมีเพียงสามสิ่งเท่านั้น: การระบุแหล่งที่มา (กุญแจนี้ได้ลงลายเซ็นเนื้อหานี้), ความสมบูรณ์ (เนื้อหาไม่ได้เปลี่ยนแปลงตั้งแต่ลงชื่อ), และการเรียงลำดับ (ใบเสร็จนี้ออกหลังใบเสร็จนั้น) พวกมันไม่พิสูจน์ว่าแอคชันของเอเจนต์ถูกต้อง นโยบายที่ชื่อใน `policy_id` ถูกประเมินจริง หรือว่าเอเจนต์ปฏิบัติตามกฎทุกข้อ ใบเสร็จเป็นรากฐาน การบริหารจัดการคือระบบที่คุณสร้างขึ้นจากรากฐานนี้

อ่าน README ของบทเรียนอีกครั้งโดยยึดขอบเขตนี้เป็นแนวทาง ความผิดพลาดที่พบบ่อยที่สุดที่ทีมทำกับใบเสร็จคือการสมมุติว่า "เรามีใบเสร็จ" หมายความว่า “เราถูกควบคุม” แต่มันไม่ใช่ ใบเสร็จทำให้พฤติกรรมของเอเจนต์สามารถตรวจสอบได้ แต่ไม่ได้ทำให้พฤติกรรมนั้นถูกต้อง


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
